# Alpamayo 1.5: Predicted CoT and Trajectory Before vs After a Weight Bit-Flip

Mirrors the structure of `inference_cam_num.ipynb`, but instead of varying the
camera configuration, we hold the input fixed and vary a **single-bit fault**
in one expert-denoiser weight. Two rollouts share the same CUDA seed so the
observed delta is dominated by the injected fault rather than sampling noise.

Pipeline:

1. **Clean baseline** rollout — capture `(CoT text, pred_xyz)`.
2. Pick one BF16 weight inside `model.expert` and flip a single bit using
   `bfa_utils.bf16_flip_one`.
3. **Faulty** rollout under the same seed.
4. Restore the weight — always, via `try/finally`, even if the faulty rollout
   raises or produces NaN trajectories.
5. Print both CoTs and plot both trajectory distributions on a shared BEV axis.

Utilities used: `bfa_utils.bf16_flip_one`, `bfa_utils.restore_one`,
`bfa_utils.collect_target_linears`, `bfa_utils.run_rollout`,
`bfa_utils.compute_traj_metrics`.

In [ ]:
import os
import mediapy as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa   # sits next to this notebook

In [ ]:
# Pin to a specific GPU. Edit GPU_ID per session — never rely on cuda:0
# defaults on the shared 8×H20 box.
GPU_ID = int(os.environ.get("ALPAMAYO_GPU", "0"))
DEVICE = torch.device(f"cuda:{GPU_ID}")
print(f"Using device: {DEVICE} (visible CUDA devices: {torch.cuda.device_count()})")

### Load model and construct data preprocessor

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to(DEVICE)
model.eval()
processor = helper.get_processor(model.tokenizer)

### Load a sample and build the prompt

Uses the same default clip and the default (4-camera) configuration that the
stock `inference.ipynb` uses.

In [ ]:
clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
clip_id = clip_ids[774]
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    DEVICE,
)
print("sequence length:", tuple(inputs.input_ids.shape))

### Show camera frames

In [ ]:
mp.show_images(
    data["image_frames"].flatten(0, 1).permute(0, 2, 3, 1),
    columns=4,
    width=200,
)

### Rollout configuration

Shared by the clean and the faulty rollout. Matches the defaults used in
`inference_cam_num.ipynb` so the trajectory distributions are directly
comparable to that notebook's outputs.

In [ ]:
ROLL_KWARGS = dict(
    top_p=0.98,
    temperature=0.6,
    num_traj_samples=16,
    max_generation_length=256,
)
SEED = 42

### Clean baseline rollout

In [ ]:
clean = bfa.run_rollout(model, model_inputs, device=DEVICE, seed=SEED, **ROLL_KWARGS)
print("clean pred_xyz:", tuple(clean["pred_xyz"].shape))
print("clean CoT length:", len(clean["cot"]), "chars")

### Choose the weight to corrupt

We pick a **middle-layer attention V projection** inside `model.expert`.
Attention V is canonically the most sensitive attack surface per the
BFA literature (`docs/sdc/pytorch_sdc_experiments_summary.md` §1): errors
propagate linearly through attention because softmax clips Q/K corruption
but not V.

Tweak `TARGET_NAME`, `BIT`, or `FLAT_IDX` in the next cell to try other
attack surfaces. The BF16 bit layout is:

| bits | field                                                  |
|------|--------------------------------------------------------|
| 0–6  | mantissa (LSB…MSB) — bounded relative error            |
| 7–13 | exponent (LSB…almost-MSB) — for \|w\|<2, mostly benign |
| 14   | exponent MSB — **catastrophic** for \|w\|<2             |
| 15   | sign                                                   |

In [ ]:
targets = bfa.collect_target_linears(model)
print(f"{len(targets)} target Linear modules in expert + action projections")

v_proj_names = [n for n in targets if n.endswith(".self_attn.v_proj")]
assert v_proj_names, "no v_proj targets found — unexpected expert structure"

# Middle-ish layer by default.
TARGET_NAME = v_proj_names[len(v_proj_names) // 2]
BIT         = 14           # exponent MSB; try 6 for mantissa MSB, 15 for sign
FLAT_IDX    = 123_456      # any coord in the weight tensor's flat index space

module = targets[TARGET_NAME]
w = module.weight.data
assert 0 <= FLAT_IDX < w.numel(), (
    f"FLAT_IDX={FLAT_IDX} out of range for {TARGET_NAME} with {w.numel()} elements"
)
pre_val = w.view(-1)[FLAT_IDX].float().item()
print(f"target     : {TARGET_NAME}")
print(f"shape      : {tuple(w.shape)}  dtype: {w.dtype}")
print(f"bit        : {BIT}  ({'sign' if BIT==15 else 'exp' if BIT>=7 else 'mantissa'})")
print(f"flat_idx   : {FLAT_IDX}")
print(f"pre-flip w : {pre_val:.6g}")

### Inject, faulty rollout, restore

All three steps are in one cell with `try/finally`. If the faulty rollout
raises or produces NaN, the weight is still restored so downstream cells
(and subsequent experiments) see a clean model.

In [ ]:
orig, flipped = bfa.bf16_flip_one(w, flat_idx=FLAT_IDX, bit=BIT)
post_val = w.view(-1)[FLAT_IDX].float().item()
print(f"post-flip w: {post_val}  (was {float(orig.float()):.6g})")

try:
    faulty = bfa.run_rollout(model, model_inputs, device=DEVICE, seed=SEED, **ROLL_KWARGS)
finally:
    bfa.restore_one(w, FLAT_IDX, orig)
    restored = w.view(-1)[FLAT_IDX].float().item()
    print(f"restored w : {restored:.6g}  (matches orig: {restored == float(orig.float())})")

print("faulty pred_xyz:", tuple(faulty["pred_xyz"].shape))
print("faulty CoT length:", len(faulty["cot"]), "chars")

### Compare Chain-of-Thought outputs

In [ ]:
bar = "=" * 70
print(bar)
print("Chain-of-Thought — clean baseline")
print(bar)
print(clean["cot"] if clean["cot"] else "(empty)")
print()
print(bar)
print(f"Chain-of-Thought — after bit-flip")
print(f"  ({TARGET_NAME}, bit={BIT}, flat_idx={FLAT_IDX})")
print(bar)
print(faulty["cot"] if faulty["cot"] else "(empty)")
print()
print(f"CoT changed: {clean['cot'] != faulty['cot']}")
print(f"length delta: {len(faulty['cot']) - len(clean['cot']):+d} chars")

### Compare predicted trajectories

Faint lines are individual sampled trajectories; the red line is the recorded
ground-truth future. A catastrophic bit-flip may push some samples to NaN —
those are filtered before plotting and counted in the note.

In [ ]:
gt_xy = data["ego_future_xyz"].cpu()[0, 0, :, :2].numpy()       # (T, 2)
clean_xy_all  = clean["pred_xyz"][0, 0, :, :, :2].numpy()       # (K, T, 2)
faulty_xy_all = faulty["pred_xyz"][0, 0, :, :, :2].numpy()

def _finite(arr):
    return arr[~np.isnan(arr.reshape(arr.shape[0], -1)).any(axis=1)]

clean_xy  = _finite(clean_xy_all)
faulty_xy = _finite(faulty_xy_all)
n_nan = faulty_xy_all.shape[0] - faulty_xy.shape[0]
if n_nan:
    print(f"note: {n_nan}/{faulty_xy_all.shape[0]} faulty samples contained NaN "
          "waypoints — filtered from the plot.")

fig, ax = plt.subplots(figsize=(7, 7))
for xy in clean_xy:
    ax.plot(xy[:, 0], xy[:, 1], "o-", color="tab:blue",   alpha=0.25, markersize=3)
for xy in faulty_xy:
    ax.plot(xy[:, 0], xy[:, 1], "o-", color="tab:orange", alpha=0.25, markersize=3)

# Legend proxies (don't clutter with per-sample legend entries).
ax.plot([], [], "o-", color="tab:blue",   label=f"clean ({len(clean_xy)} samples)")
ax.plot([], [], "o-", color="tab:orange", label=f"after bit-flip ({len(faulty_xy)} samples)")
ax.plot(gt_xy[:, 0], gt_xy[:, 1], "r-", linewidth=2.5, label="ground truth")

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(
    f"Trajectories before vs after weight bit-flip\n"
    f"({TARGET_NAME}  bit={BIT}  flat_idx={FLAT_IDX})"
)
ax.legend(loc="best", fontsize=8)
ax.axis("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Quantitative summary

Aggregate ADE/FDE over the 16 samples in each condition, plus a paired
waypoint-shift distribution — for each sample index `k`, how far did the
faulty rollout diverge from the clean one?

In [ ]:
clean_m  = bfa.compute_traj_metrics(clean_xy_all,  gt_xy)
faulty_m = bfa.compute_traj_metrics(faulty_xy_all, gt_xy)

print(f"{'metric':<12}  {'clean':>12}  {'faulty':>12}  {'delta':>12}")
print("-" * 54)
for k in ["n_finite", "minADE_m", "meanADE_m", "minFDE_m", "meanFDE_m"]:
    cv, fv = clean_m[k], faulty_m[k]
    if k == "n_finite":
        print(f"{k:<12}  {cv:>12d}  {fv:>12d}  {'':>12}")
    else:
        d = fv - cv
        print(f"{k:<12}  {cv:>12.6f}  {fv:>12.6f}  {d:>+12.6f}")

# Paired per-sample drift: clean[k] vs faulty[k] when both are finite.
pair_mask = (
    ~np.isnan(clean_xy_all.reshape(clean_xy_all.shape[0], -1)).any(axis=1)
    & ~np.isnan(faulty_xy_all.reshape(faulty_xy_all.shape[0], -1)).any(axis=1)
)
if pair_mask.any():
    shift = np.linalg.norm(
        clean_xy_all[pair_mask] - faulty_xy_all[pair_mask], axis=-1
    )   # (n_ok, T)
    print()
    print(f"per-sample waypoint shift  (n_ok = {int(pair_mask.sum())}):")
    print(f"  max shift across any (sample, waypoint): {shift.max():.3f} m")
    print(f"  mean shift:                              {shift.mean():.3f} m")
    print(f"  shift at final waypoint (mean):          {shift[:, -1].mean():.3f} m")
else:
    print("\n(no sample indices have finite trajectories in both conditions)")